# 02 — Embedding Generation & Hierarchical Clustering

**Goal:** Convert cleaned descriptions to semantic embeddings, discover natural topic groupings via hierarchical clustering, and visualize the results.

**Key decisions:**
- `all-MiniLM-L6-v2`: 22M params, 384-dim, 5× faster than MPNet, only 2-3% quality drop
- Ward's linkage: balanced clusters (vs Average's imbalance, Complete's fragmentation)
- K=7: chosen for interpretability over K=15's higher silhouette score

## 1. Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

# Load cleaned descriptions from notebook 01
df = pd.read_csv('../data/cleaned/dataset3_cleaned.csv')
print(f'Loaded {len(df)} descriptions')
descriptions = df['cleaned_description'].tolist()

## 2. Load Embedding Model
**Model:** `all-MiniLM-L6-v2`
- 22M parameters (vs 110M for all-mpnet-base-v2)
- 384-dim embeddings (sweet spot: rich enough, compact enough)
- 5× faster inference on CPU
- Only 2-3% drop in benchmark performance

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model max sequence length: {model.max_seq_length}')
print(f'Output dimension: {model.get_sentence_embedding_dimension()}')

## 3. Generate Embeddings
**Parameter choices:**
- `batch_size=32`: Balanced for CPU — 32 is optimal for CPU-only inference
- `show_progress_bar=True`: Monitor processing time for large datasets (~100-200 samples/sec on modern CPU)

In [ ]:
print('Generating embeddings...')
embeddings = model.encode(descriptions, show_progress_bar=True, batch_size=32)
print(f'Embeddings shape: {embeddings.shape}')  # Expected: (n_samples, 384)

## 4. Dendrogram — Visual Cluster Exploration
Before committing to a K value, visualize the hierarchical structure.
- Long vertical lines = natural separation points (large distance between clusters)
- Short vertical lines = similar clusters being merged (too granular)
- `truncate_mode='lastp', p=30`: Shows only the last 30 merges (readable)

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

linkage_matrix = linkage(embeddings, method='ward')

plt.figure(figsize=(15, 7))
dendrogram(linkage_matrix, truncate_mode='lastp', p=30)
plt.title('Hierarchical Clustering Dendrogram (Last 30 merges)')
plt.xlabel('Cluster Size')
plt.ylabel('Distance (Ward)')
plt.tight_layout()
plt.show()

## 5. Linkage Method Comparison
**Experimental results — three linkage methods tested:**

| Method | Result | Decision |
|--------|--------|----------|
| **Ward** | Balanced clusters, low intra-cluster variance | ✅ **CHOSEN** |
| Average | 60% samples in one giant cluster | ❌ Rejected (imbalanced) |
| Complete | Many singletons, over-fragmented | ❌ Rejected (too sensitive) |

**Ward's method** minimizes variance within clusters — best for cohesive topic groups.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

# Test all three linkage methods
for method in ['ward', 'average', 'complete']:
    clusterer = AgglomerativeClustering(n_clusters=7, linkage=method)
    labels = clusterer.fit_predict(embeddings)
    counts = pd.Series(labels).value_counts().sort_index()
    print(f'{method:10s}: {counts.tolist()}')

## 6. K Selection — Silhouette Analysis
We evaluate K from 5 to 20 using silhouette score, then make a **domain-driven decision**.

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []
K_range = range(5, 20)

for k in K_range:
    clusterer = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = clusterer.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels)
    silhouette_scores.append(score)
    print(f'K={k}: Silhouette Score = {score:.4f}')

# Plot
plt.figure(figsize=(10, 5))
plt.plot(K_range, silhouette_scores, marker='o', linewidth=2)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis for Optimal K')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. K Selection — Domain Decision
**Critical insight:** K=15 has higher silhouette score, but we choose K=7.

**Why K=7 over K=15?**
1. **Interpretability:** 15 topics too granular for maintenance engineers
2. **Diminishing returns:** Beyond K=7, clusters split existing topics rather than find new patterns
3. **Business alignment:** K=7 maps to natural failure categories (pumps, valves, piping, electrical, corrosion, operational, preventive)
4. **Data per cluster:** Minimum 449 samples — sufficient for downstream T5 fine-tuning
5. **Marginal improvement:** 0.4267 → 0.4891 is only +0.06

> **Lesson:** Don't blindly optimize metrics. Domain knowledge + interpretability > marginal silhouette improvements.

## 8. Final Cluster Assignment (K=7)

In [ ]:
clustering = AgglomerativeClustering(n_clusters=7, linkage='ward')
df['cluster'] = clustering.fit_predict(embeddings)

print('Cluster distribution:')
print(df['cluster'].value_counts().sort_index())

## 9. t-SNE Visualization
**Parameter choices:**
- `perplexity=30`: Standard for ~5000 samples (rule of thumb: 5 to 50)
- `n_iter=1000`: Sufficient convergence (observed ~600 iterations needed)
- `init='pca'`: Faster convergence + more reproducible than random init

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000, init='pca')
embeddings_2d = tsne.fit_transform(embeddings)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
                      c=df['cluster'], cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter, label='Cluster')
plt.title('t-SNE Visualization of Clusters (K=7, Ward linkage)')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.tight_layout()
plt.show()

## 10. Manual Cluster Inspection & Topic Labeling
Read 10 random samples from each cluster to identify common themes.

In [ ]:
for cluster_id in range(7):
    print(f"\n{'='*60}")
    print(f'Cluster {cluster_id}')
    print('='*60)
    samples = df[df['cluster'] == cluster_id]['cleaned_description'].sample(5, random_state=42)
    for i, text in enumerate(samples, 1):
        print(f'  {i}. {text}')

## 11. Topic Mapping
Based on manual inspection, we assign human-readable labels to each cluster.

In [ ]:
topic_map = {
    0: 'Pump/Mechanical Failures',
    1: 'Valve Operations',
    2: 'Leakage/Corrosion',
    3: 'Electrical/Instrumentation',
    4: 'Piping/Structural',
    5: 'Operational Issues',
    6: 'Preventive Maintenance'
}

df['topic'] = df['cluster'].map(topic_map)

# Show full distribution
print(df['topic'].value_counts().to_string())

# Save for next notebook
df.to_csv('../data/cleaned/dataset3_labeled.csv', index=False)
print(f'\n✅ Labeled dataset saved: ../data/cleaned/dataset3_labeled.csv')